### Задание на Четверку (Проведение исследований моделями семантической сегментации)

#### 4. Имплементация алгоритма машинного обучения

улучшим наши имплементированные модели: добавим наши гипотезы, которые мы рассматривали в пункте 3

импорты

In [13]:
import os
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

from tqdm import tqdm

пути

In [14]:
IMAGES_DIR = "../train_images"
MASKS_DIR = "../train_labels"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset

In [15]:
class SegmentationDataset(Dataset):
    def __init__(self, images_dir, masks_dir, transform=None):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        self.transform = transform

        self.images = sorted([
            f for f in os.listdir(images_dir)
            if f.lower().endswith((".jpg", ".png", ".jpeg"))
        ])

    def __len__(self):
        return len(self.images)

    def find_mask(self, name):
        for ext in [".png", ".jpg", ".jpeg"]:
            path = os.path.join(self.masks_dir, name + ext)
            if os.path.exists(path):
                return path
        return None

    def __getitem__(self, idx):
        img_name = self.images[idx]
        name = os.path.splitext(img_name)[0]

        img_path = os.path.join(self.images_dir, img_name)
        mask_path = self.find_mask(name)

        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 0).astype(np.float32)

        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image = aug["image"]
            mask = aug["mask"]

        mask = torch.tensor(mask).unsqueeze(0).float()

        return image, mask

аугментации

In [16]:
def get_transforms():
    return A.Compose([
        A.Resize(256, 256),

        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),

        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=25, p=0.5),

        A.RandomBrightnessContrast(p=0.5),
        A.GaussNoise(p=0.2),
        A.Blur(p=0.2),

        A.Normalize(),
        ToTensorV2()
    ])

улучшенная CNN (U-Net light)

In [17]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.enc1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU()
        )

        self.pool = nn.MaxPool2d(2)

        self.enc2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU()
        )

        self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)

        self.dec = nn.Sequential(
            nn.Conv2d(96, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 1, 1)
        )

    def forward(self, x):
        x1 = self.enc1(x)
        x2 = self.pool(x1)

        x3 = self.enc2(x2)
        x4 = self.up(x3)

        x = torch.cat([x4, x1], dim=1)
        return self.dec(x)

2. Простой Transformer (очень упрощённый ViT для сегментации)

In [18]:
import torch
import torch.nn as nn

class SimpleViT(nn.Module):
    def __init__(self, img_size=256, patch_size=16, in_ch=3, dim=128, dropout=0.1):
        super().__init__()

        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2

        self.patch_embed = nn.Conv2d(in_ch, dim, patch_size, patch_size)

        self.pos_embed = nn.Parameter(torch.randn(1, self.num_patches, dim))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=dim,
            nhead=4,
            dim_feedforward=dim * 4,
            dropout=dropout,
            batch_first=True,  
            norm_first=True
        )

        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)

        self.decoder = nn.Sequential(
            nn.Conv2d(dim, dim // 2, 3, padding=1),
            nn.ReLU(),
            nn.Upsample(scale_factor=patch_size, mode="bilinear", align_corners=False),
            nn.Conv2d(dim // 2, 32, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 1, 1)
        )

    def forward(self, x):
        x = self.patch_embed(x)         

        b, c, h, w = x.shape

        x = x.flatten(2).transpose(1, 2)  

        x = x + self.pos_embed[:, :x.size(1), :]

        x = self.transformer(x)

        x = x.transpose(1, 2).reshape(b, c, h, w)

        return self.decoder(x)

Loss + метрики

In [19]:
loss_fn = nn.BCEWithLogitsLoss()

def iou(pred, target):
    pred = (pred > 0.5).float()
    inter = (pred * target).sum()
    union = pred.sum() + target.sum() - inter
    return (inter + 1e-6) / (union + 1e-6)


def dice(pred, target):
    pred = (pred > 0.5).float()
    inter = (pred * target).sum()
    return (2 * inter + 1e-6) / (pred.sum() + target.sum() + 1e-6)


def acc(pred, target):
    pred = (pred > 0.5).float()
    return (pred == target).float().mean()

Train loop

In [20]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        preds = model(images)

        if masks.ndim == 3:
            masks = masks.unsqueeze(1)

        loss = loss_fn(preds, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

Evaluation

In [21]:
def evaluate(model, loader):
    model.eval()

    iou_total = 0
    dice_total = 0
    acc_total = 0

    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)

            preds = torch.sigmoid(model(images))

            iou_total += iou(preds, masks).item()
            dice_total += dice(preds, masks).item()
            acc_total += acc(preds, masks).item()

    n = len(loader)

    return {
        "IoU": iou_total / n,
        "Dice": dice_total / n,
        "Acc": acc_total / n
    }

подбор гиперпарамтеров

In [22]:
from torch.utils.data import random_split, DataLoader
import torch

base_dataset = SegmentationDataset(IMAGES_DIR, MASKS_DIR, transform=None)

train_size = int(0.8 * len(base_dataset))
val_size = len(base_dataset) - train_size

train_subset, val_subset = random_split(
    base_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

class WrappedDataset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, mask = self.subset[idx]

        img = img.permute(1, 2, 0).numpy() if torch.is_tensor(img) else img
        mask = mask.squeeze().numpy() if torch.is_tensor(mask) else mask

        if self.transform:
            aug = self.transform(image=img, mask=mask)
            img = aug["image"]
            mask = aug["mask"]

        mask = torch.tensor(mask).float().unsqueeze(0)

        return img, mask
    
train_dataset = WrappedDataset(train_subset, get_transforms())

val_dataset = WrappedDataset(
    val_subset,
    A.Compose([
        A.Resize(256, 256),
        A.Normalize(),
        ToTensorV2()
    ])
)

def get_loaders(batch_size):
    return (
        DataLoader(train_dataset, batch_size=batch_size, shuffle=True),
        DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    )

train_loader, val_loader = get_loaders(8)

CNN

In [23]:
model_cnn = SimpleCNN().to(device)
opt_cnn = torch.optim.Adam(model_cnn.parameters(), lr=1e-3)

for epoch in range(10):
    loss = train_one_epoch(model_cnn, train_loader, opt_cnn)
    print(f"CNN Epoch {epoch+1}: {loss:.4f}")

C:\Users\2b100\AppData\Local\Temp\ipykernel_14748\138741405.py:34: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask = torch.tensor(mask).float().unsqueeze(0)


CNN Epoch 1: 0.4752
CNN Epoch 2: 0.3766
CNN Epoch 3: 0.3740
CNN Epoch 4: 0.3325
CNN Epoch 5: 0.3698
CNN Epoch 6: 0.3519
CNN Epoch 7: 0.3528
CNN Epoch 8: 0.3399
CNN Epoch 9: 0.3518
CNN Epoch 10: 0.3442


Transformer

In [24]:
model_vit = SimpleViT().to(device)
opt_vit = torch.optim.Adam(model_vit.parameters(), lr=1e-3)

for epoch in range(10):
    loss = train_one_epoch(model_vit, train_loader, opt_vit)
    print(f"ViT Epoch {epoch+1}: {loss:.4f}")

C:\Users\2b100\AppData\Local\Temp\ipykernel_14748\3087000436.py:24: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=4)
C:\Users\2b100\AppData\Local\Temp\ipykernel_14748\138741405.py:34: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask = torch.tensor(mask).float().unsqueeze(0)


ViT Epoch 1: 0.5080
ViT Epoch 2: 0.3719
ViT Epoch 3: 0.2668
ViT Epoch 4: 0.2406
ViT Epoch 5: 0.1702
ViT Epoch 6: 0.1792
ViT Epoch 7: 0.1999
ViT Epoch 8: 0.1830
ViT Epoch 9: 0.1786
ViT Epoch 10: 0.1987


Оценка качества

In [25]:
cnn_metrics = evaluate(model_cnn, val_loader)
vit_metrics = evaluate(model_vit, val_loader)

print("CNN:", cnn_metrics)
print("ViT:", vit_metrics)

C:\Users\2b100\AppData\Local\Temp\ipykernel_14748\138741405.py:34: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask = torch.tensor(mask).float().unsqueeze(0)


CNN: {'IoU': 0.6845991611480713, 'Dice': 0.8108851462602615, 'Acc': 0.9030265808105469}
ViT: {'IoU': 0.812184602022171, 'Dice': 0.8961420059204102, 'Acc': 0.9405811578035355}


In [26]:
import pandas as pd

results = pd.DataFrame([cnn_metrics, vit_metrics], index=["CNN", "ViT"])
print(results)

          IoU      Dice       Acc
CNN  0.684599  0.810885  0.903027
ViT  0.812185  0.896142  0.940581


### Анализ результатов и выводы

Проведённые эксперименты показали, что применение улучшенного бейзлайна (аугментации данных, подбор гиперпараметров и оптимизация обучения) приводит к значительному росту качества моделей. Однако библиотечные реализации стабильно превосходят имплементированные модели по всем метрикам.

Наибольший разрыв наблюдается у CNN-архитектуры, в то время как Vision Transformer демонстрирует более близкие к улучшенному baseline результаты, что подтверждает его большую устойчивость к изменениям в реализации. Тем не менее, все модели демонстрируют корректное обучение и адекватное качество сегментации.